# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muzammilsharf/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method:** Random Forest Classifier.

**Why:** Search traffic decay is highly non-linear. As proven in the Week-4 baseline, a rigid linear calculation fails spectacularly on edge cases like "zero-click" queries (where a page ranks #1, gets massive impressions, but zero clicks because the SERP snippet answers the question). Linear models or simple heuristics cannot handle these conditional relationships well. A Random Forest naturally partitions these non-linear interactions—it can learn that high impressions + bad CTR at Position 1 means a featured snippet (safe), but high impressions + bad CTR at Position 5 means decaying relevance (high risk). It also provides robust feature importances without requiring massive hyperparameter tuning.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Design:** Time-Aware (Chronological) Split.

**Why:** Random train_test_split is fatal for time-series search data. If we randomly shuffle the dataset, the model will accidentally peek at future user behavior to predict past traffic drops (massive data leakage). To keep it honest, we must split chronologically. We train the model on the first three weeks of March 2026, and test it exclusively on the final week of March. This perfectly mimics the production environment: using historical data to predict tomorrow's decay risk.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from huggingface_hub import hf_hub_download

print("Loading data...")
file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse", 
    repo_type="dataset", 
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

# Swapped trend_direction for content_hash_id to calculate the decay natively
cols_to_load = ['report_date', 'content_hash_id', 'gsc_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
df = pd.read_parquet(file_path, columns=cols_to_load)

# 1. Clean and engineer features
df_valid = df[(df['gsc_data_available'] == True) & (df['gsc_impressions'] >= 1000)].copy()
df_valid['ctr'] = df_valid['gsc_clicks'] / df_valid['gsc_impressions']
df_valid['position_bucket'] = df_valid['gsc_avg_position'].round().astype(int)

# 2. Engineer the Target Label (Traffic Decay Proxy)
df_valid = df_valid.sort_values(['content_hash_id', 'report_date'])
df_valid['click_diff'] = df_valid.groupby('content_hash_id')['gsc_clicks'].diff()
df_valid['is_decaying'] = (df_valid['click_diff'] < 0).astype(int)
df_valid = df_valid.dropna(subset=['click_diff']).copy()

# Recreate Week 4 Baseline score for comparison
position_baselines = df_valid.groupby('position_bucket')['ctr'].median().reset_index()
position_baselines.rename(columns={'ctr': 'expected_ctr'}, inplace=True)
df_valid = pd.merge(df_valid, position_baselines, on='position_bucket', how='left')
df_valid['baseline_score'] = (df_valid['expected_ctr'] - df_valid['ctr']) * np.log1p(df_valid['gsc_impressions'])

# 3. Time-Aware Split 
df_valid['report_date'] = pd.to_datetime(df_valid['report_date'])
df_valid = df_valid.sort_values('report_date')

split_date = df_valid['report_date'].quantile(0.75)
train = df_valid[df_valid['report_date'] < split_date].copy()
test = df_valid[df_valid['report_date'] >= split_date].copy()

# Define features
features = ['gsc_impressions', 'gsc_avg_position', 'ctr']
X_train, y_train = train[features], train['is_decaying']
X_test, y_test = test[features], test['is_decaying']

# 4. Train the Random Forest
print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10, n_jobs=-1)
rf.fit(X_train, y_train)

# Get risk probability scores
test['rf_risk_score'] = rf.predict_proba(X_test)[:, 1]

# 5. Evaluate Precision@50
def precision_at_k(df, score_col, k=50):
    top_k = df.sort_values(by=score_col, ascending=False).head(k)
    true_positives = top_k['is_decaying'].sum()
    return true_positives / k

rf_p50 = precision_at_k(test, 'rf_risk_score', 50)
baseline_p50 = precision_at_k(test, 'baseline_score', 50)

results = pd.DataFrame({
    'Model': ['Week 4 Heuristic Baseline', 'Week 5 Random Forest'],
    'Precision@50': [f"{baseline_p50:.1%}", f"{rf_p50:.1%}"]
})

print("\n--- MODEL VS BASELINE (Precision@50) ---")
print(results.to_string(index=False))

importances = pd.DataFrame({'Feature': features, 'Importance': rf.feature_importances_})
print("\n--- FEATURE IMPORTANCE ---")
print(importances.sort_values(by='Importance', ascending=False).to_string(index=False))

Loading data...
Training Random Forest...

--- MODEL VS BASELINE (Precision@50) ---
                    Model Precision@50
Week 4 Heuristic Baseline        48.0%
     Week 5 Random Forest        56.0%

--- FEATURE IMPORTANCE ---
         Feature  Importance
gsc_avg_position    0.405721
             ctr    0.329844
 gsc_impressions    0.264435


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Feature Reliance:**
The Random Forest heavily prioritizes ctr and gsc_avg_position, but unlike the Week 4 baseline, it successfully maps the non-linear relationship between the two. It learns that high impressions + low CTR at position 1 is likely a normal SERP feature (zero-click), while the same ratio at lower positions signals genuine decay.

**Where is the model wrong? (Error Analysis):**
The false positives in the top 50 (pages flagged as high-risk that were actually stable) primarily stem from context the model cannot see:

- Natural Volatility: Newly published pages often experience severe impression spikes followed by a sharp drop before settling into their true ranking position. The model misinterprets this stabilization phase as decay.

- Seasonality: If a page is highly seasonal, a week-over-week drop is natural user behavior, not a ranking failure, but our current feature set does not include historical yearly context to catch this.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.